# SciVer Full-Search v3 server notebook

This is the canonical thin Milestone 6 operator notebook. It delegates deterministic preparation, preflight, SEARCH, freezing, and paired FINAL execution to `meta_harness.full_search_v3_server`. It contains no experiment business logic and is safe to restart with the same workspace and run ID.

Run cells in order. SMOKE, FULL_SEARCH, and FINAL are independently disabled by default. Never save notebook outputs after entering runtime credentials.

## 1. Non-secret operator settings

Edit paths and identifiers for the server. The commit must be a pinned 40-character Git SHA; do not use a branch name. The solver model is displayed for operator confirmation and is enforced by the repository's locked configuration.

In [ ]:
from pathlib import Path

GIT_REPOSITORY_URL = "REPLACE_WITH_GIT_REPOSITORY_URL"
PINNED_COMMIT_SHA = "0000000000000000000000000000000000000000"
WORKSPACE_DIRECTORY = Path("/server/workspace/sciver-full-search-v3")
DATASET_PATH = Path("/server/data/sciver/testset.json")
RUN_ID = "sciver-full-search-v3-run-001"
SOLVER_MODEL_ID = "Qwen/Qwen3.5-35B-A3B"
CONFIG_PATH = None  # None uses the locked canonical V3 configuration.

REPOSITORY_DIRECTORY = WORKSPACE_DIRECTORY / "repository"
PREPARATION_DIRECTORY = WORKSPACE_DIRECTORY / "runs" / RUN_ID / "preparation"
RUN_SMOKE = False
RUN_FULL_SEARCH = False
RUN_FINAL = False

## 2. Clone when absent, or verify the existing checkout

In [ ]:
import subprocess

WORKSPACE_DIRECTORY.mkdir(parents=True, exist_ok=True)
if not REPOSITORY_DIRECTORY.exists():
    subprocess.run(["git", "clone", GIT_REPOSITORY_URL, str(REPOSITORY_DIRECTORY)], check=True)
elif not (REPOSITORY_DIRECTORY / ".git").is_dir():
    raise RuntimeError(f"Existing path is not a Git checkout: {REPOSITORY_DIRECTORY}")

assert (REPOSITORY_DIRECTORY / "AGENTS.md").is_file()
print(f"Repository directory ready: {REPOSITORY_DIRECTORY}")

## 3. Fetch and check out the exact pinned commit

This never advances to a changing branch tip. An existing checkout must be clean before the detached checkout.

In [ ]:
if len(PINNED_COMMIT_SHA) != 40 or any(character not in "0123456789abcdef" for character in PINNED_COMMIT_SHA):
    raise ValueError("PINNED_COMMIT_SHA must be a 40-character lowercase Git SHA")

status = subprocess.run(["git", "status", "--porcelain"], cwd=REPOSITORY_DIRECTORY, check=True, capture_output=True, text=True)
if status.stdout.strip():
    raise RuntimeError("Existing checkout has local changes; use a clean checkout or choose another workspace")
subprocess.run(["git", "fetch", "--tags", "origin", PINNED_COMMIT_SHA], cwd=REPOSITORY_DIRECTORY, check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT_SHA], cwd=REPOSITORY_DIRECTORY, check=True)
checked_out_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIRECTORY, check=True, capture_output=True, text=True).stdout.strip()
if checked_out_commit != PINNED_COMMIT_SHA:
    raise RuntimeError("Checkout did not resolve to the requested pinned commit")
print(f"Checked out pinned commit: {checked_out_commit}")

## 4. Install project dependencies into this Jupyter environment

In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPOSITORY_DIRECTORY / "requirements.txt")], check=True)

## 5. Verify server readiness

This reports only local environment metadata, repository identity, CLI availability, disk space, and required-file existence.

In [ ]:
import importlib.util
import shutil

required_files = [
    REPOSITORY_DIRECTORY / "AGENTS.md",
    REPOSITORY_DIRECTORY / "requirements.txt",
    REPOSITORY_DIRECTORY / "meta_harness" / "full_search_v3_server.py",
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Required repository files are missing: {missing_files}")
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Dataset path does not exist: {DATASET_PATH}")

gpu_visible = importlib.util.find_spec("torch") is not None
gpu_summary = "not checked (torch is unavailable)"
if gpu_visible:
    import torch
    gpu_summary = {"available": torch.cuda.is_available(), "count": torch.cuda.device_count()}

readiness = {
    "python": sys.version.split()[0],
    "repository_commit": checked_out_commit,
    "solver_model_id": SOLVER_MODEL_ID,
    "codex_cli_available": shutil.which("codex") is not None,
    "gpu": gpu_summary,
    "free_disk_gib": round(shutil.disk_usage(WORKSPACE_DIRECTORY).free / (1024 ** 3), 2),
}
if not readiness["codex_cli_available"]:
    raise RuntimeError("Codex CLI is required for SEARCH proposal generation and is not on PATH")
readiness

## 6. Import the repository M6 interface and locate the source dataset

In [ ]:
if str(REPOSITORY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIRECTORY))

from meta_harness.full_search_v3_server import (
    freeze_full_search_v3_server_winner,
    inspect_full_search_v3_server_final_status,
    inspect_full_search_v3_server_status,
    preflight_full_search_v3_server_final,
    preflight_full_search_v3_server_run,
    prepare_full_search_v3_server_run,
    run_full_search_v3_server_smoke,
    solver_identity_from_api_url,
    start_or_resume_full_search_v3_server_final,
    start_or_resume_full_search_v3_server_run,
)

dataset_location = {"dataset_path": str(DATASET_PATH), "exists": DATASET_PATH.is_file()}
dataset_location

## 7. Prepare or reuse the deterministic configured split

M1 validates the source and atomically creates or verifies the exact paper-disjoint split. The notebook never reads or displays examples, labels, IDs, images, or membership lists.

In [ ]:
prepared = prepare_full_search_v3_server_run(
    dataset_path=DATASET_PATH,
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    preparation_directory=PREPARATION_DIRECTORY,
    config_path=CONFIG_PATH,
)
{
    "run_id": prepared["run_id"],
    "split_sha256": prepared["split_sha256"],
    "preparation_identity_sha256": prepared["preparation_identity_sha256"],
    "search": prepared["SEARCH"],
    "final": prepared["FINAL"],
    "sample_overlap_count": prepared["sample_overlap_count"],
    "paper_overlap_count": prepared["paper_overlap_count"],
    "search_artifact_directory": str(Path(prepared["search_safe_manifest_path"]).parent),
}

## 8. Collect runtime API values without displaying them

The URL and key remain process-local variables. They are not printed, placed in configuration, or returned by any displayed dictionary. Use a hidden prompt for the key.

In [ ]:
import os
from getpass import getpass

api_url = os.environ.get("API_URL") or input("API_URL (runtime only): " )
api_key = os.environ.get("API_KEY") or getpass("API_KEY (runtime only): ")
if not api_url.strip() or not api_key.strip():
    raise RuntimeError("Both API_URL and API_KEY are required for the live stages")
solver_identity_sha256 = solver_identity_from_api_url(api_url)
print("Runtime API values accepted without display.")

## 9. Offline SEARCH preflight

This validates immutable identities, split counts and paper disjointness, model/generation/parser identities, logical-call budgets, checkpoint locations, and compatible resume status. It constructs no live solver or proposer client.

In [ ]:
search_preflight = preflight_full_search_v3_server_run(
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    search_safe_manifest_path=prepared["search_safe_manifest_path"],
    search_records_path=prepared["search_dataset_path"],
    source_commit=PINNED_COMMIT_SHA,
    solver_identity_sha256=solver_identity_sha256,
)
if search_preflight["solver"]["model"] != SOLVER_MODEL_ID:
    raise RuntimeError("SOLVER_MODEL_ID does not match the repository's locked V3 solver configuration")
{
    "split_sizes_and_paper_groups": {"search": prepared["SEARCH"], "final": prepared["FINAL"]},
    "disjointness": {"sample_overlap_count": prepared["sample_overlap_count"], "paper_overlap_count": prepared["paper_overlap_count"]},
    **{
        key: search_preflight[key]
        for key in ("protocol_id", "run_id", "resume", "source_commit", "config_sha256", "split_sha256", "search_membership_sha256", "canonical_p0_prompt_sha256", "solver", "parser_version", "workload", "checkpoints")
    },
}

## 10. Isolated SMOKE guard and compatible receipt

Set `RUN_SMOKE = True` only with explicit authorization for one canonical P0 SEARCH request. Its create-once receipt is isolated from production SEARCH state and is required before FULL_SEARCH can start.

In [ ]:
if RUN_SMOKE:
    smoke_status = run_full_search_v3_server_smoke(
        repository_root=REPOSITORY_DIRECTORY,
        run_id=RUN_ID,
        search_safe_manifest_path=prepared["search_safe_manifest_path"],
        search_records_path=prepared["search_dataset_path"],
        authorize_smoke_execution=True,
        api_url=api_url,
        api_key=api_key,
        source_commit=PINNED_COMMIT_SHA,
    )
    smoke_status
else:
    print("SMOKE is disabled. Set RUN_SMOKE = True only with explicit authorization.")

## 11. Explicit FULL_SEARCH guard

Set `RUN_FULL_SEARCH = True` only after reviewing preflight, creating the compatible SMOKE receipt, and receiving explicit production SEARCH authorization. Reusing this run ID resumes only compatible durable production state.

In [ ]:
if RUN_FULL_SEARCH:
    search_result = start_or_resume_full_search_v3_server_run(
        repository_root=REPOSITORY_DIRECTORY,
        run_id=RUN_ID,
        search_safe_manifest_path=prepared["search_safe_manifest_path"],
        search_records_path=prepared["search_dataset_path"],
        authorize_search_execution=True,
        api_url=api_url,
        api_key=api_key,
        source_commit=PINNED_COMMIT_SHA,
    )
    {"search": search_result["search"]}
else:
    print("FULL_SEARCH is disabled. Set RUN_FULL_SEARCH = True only after SMOKE and explicit production authorization.")

## 12. Inspect SEARCH progress and terminal status

In [ ]:
try:
    search_status = inspect_full_search_v3_server_status(repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID)
except RuntimeError:
    search_status = {"status": "not_started", "message": "SEARCH has not started; preflight remains valid."}
{
    key: search_status.get(key)
    for key in ("run_id", "status", "stop_reason", "completed_iterations", "p0_status", "winner_id", "patience", "winner_metrics", "run_identity_sha256", "message")
}

## 13. Freeze the SEARCH winner only after terminal completion

In [ ]:
frozen_winner = None
if search_status["status"] in {"patience_stopped", "max_stopped"}:
    frozen_winner = freeze_full_search_v3_server_winner(repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID)
    frozen_winner
else:
    print("SEARCH is not terminal; FINAL remains locked and no winner was frozen.")

## 14. FINAL preflight

The M6 interface keeps trusted FINAL material internal. This display contains only safe hashes and prompt identities; it never displays examples, IDs, labels, images, or model payloads.

In [ ]:
if frozen_winner is None:
    final_preflight = {"status": "locked_until_search_freeze"}
else:
    final_preflight = preflight_full_search_v3_server_final(
        repository_root=REPOSITORY_DIRECTORY,
        run_id=RUN_ID,
        dataset_path=DATASET_PATH,
        private_manifest_path=prepared["private_manifest_path"],
        search_safe_manifest_path=prepared["search_safe_manifest_path"],
        solver_identity_sha256=solver_identity_sha256,
    )
final_preflight

## 15. Separate explicit paired FINAL guard

Set `RUN_FINAL = True` only after a separate explicit authorization. This cannot modify SEARCH state or the frozen winner.

In [ ]:
if RUN_FINAL and frozen_winner is not None:
    final_result = start_or_resume_full_search_v3_server_final(
        repository_root=REPOSITORY_DIRECTORY,
        run_id=RUN_ID,
        dataset_path=DATASET_PATH,
        private_manifest_path=prepared["private_manifest_path"],
        search_safe_manifest_path=prepared["search_safe_manifest_path"],
        authorize_final_execution=True,
        api_url=api_url,
        api_key=api_key,
    )
    {"final": final_result["final"]}
else:
    print("FINAL is disabled or locked. Set RUN_FINAL = True only after a frozen winner and separate explicit authorization.")

## 16. Sanitized aggregate results and artifact locations

Before saving or sharing this notebook, use **Kernel > Restart and Clear Output**. Do not save runtime API values or outputs from cells that could include them.

In [ ]:
try:
    final_status = inspect_full_search_v3_server_final_status(repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID)
except RuntimeError:
    final_status = {"status": "not_started_or_locked"}
sanitized_results = {
    "search": {
        key: search_status[key]
        for key in ("status", "completed_iterations", "winner_id", "winner_metrics", "run_identity_sha256")
    },
    "freeze": None if frozen_winner is None else {
        key: frozen_winner[key]
        for key in ("winner_id", "prompt_variant", "artifact_sha256", "prompt_sha256", "frozen_winner_path")
    },
    "final": final_status,
    "artifacts": {
        "run_directory": search_preflight["run_directory"],
        "checkpoints": search_preflight["checkpoints"],
    },
}
sanitized_results

api_url = None
api_key = None